
# N05: LightGBM Categorical Core (Shadowcat Architecture Clone)


In [ ]:

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

ID_COL, TARGET_COL = "id", "health_condition"
CLASSES = ("at-risk", "fit", "unhealthy")
SEED = 20260712
N_FOLDS = 5


In [ ]:

# 1. Load Data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")


In [ ]:

# 2. Replicate Shadowcat Exact Feature Processing
# The shadowcat script did zero target encoding and zero imputation.
# It simply mapped the target to integers and converted object columns to pandas Categorical types.

y = train_df[TARGET_COL].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
X = train_df.drop(columns=[c for c in (ID_COL, TARGET_COL) if c in train_df.columns])
Xt = test_df.drop(columns=[c for c in (ID_COL,) if c in test_df.columns])
Xt = Xt[X.columns]

cat_cols = [c for c in X.columns if X[c].dtype == "object"]

# Strict pandas Categorical conversion using combined train/test categories
for c in cat_cols:
    cats = pd.Categorical(pd.concat([X[c], Xt[c]]).astype(str)).categories
    X[c]  = pd.Categorical(X[c].astype(str),  categories=cats)
    Xt[c] = pd.Categorical(Xt[c].astype(str), categories=cats)

print(f"Processed Features: {X.columns.tolist()}")
print(f"Categorical Columns Native to LightGBM: {cat_cols}")


In [ ]:

# 3. Model Training exactly as implemented in reference script
oof = np.zeros((len(X), 3))
tst = np.zeros((len(Xt), 3))
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for f, (tr_i, va_i) in enumerate(skf.split(X, y)):
    m = LGBMClassifier(
        objective="multiclass", 
        num_class=3, 
        class_weight="balanced",
        n_estimators=1200, 
        learning_rate=0.03, 
        num_leaves=63,
        colsample_bytree=0.8, 
        subsample=0.8, 
        subsample_freq=1,
        reg_lambda=1.0, 
        random_state=SEED + f, 
        n_jobs=-1, 
        verbose=-1,
    )
    
    # Train using LightGBM's native categorical handling
    m.fit(X.iloc[tr_i], y[tr_i],
          eval_set=[(X.iloc[va_i], y[va_i])],
          eval_metric="multi_logloss")
          
    oof[va_i] = m.predict_proba(X.iloc[va_i])
    tst += m.predict_proba(Xt) / N_FOLDS
    
    print(f"Fold {f + 1}/{N_FOLDS} accuracy={accuracy_score(y[va_i], oof[va_i].argmax(1)):.5f}")

print(f"\nOOF Overall accuracy: {accuracy_score(y, oof.argmax(1)):.5f}")


In [ ]:

# 4. Generate Predictions
sub_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].astype("int64"),
    TARGET_COL: [CLASSES[i] for i in tst.argmax(axis=1)]
})

sub_df.to_csv("submission_shadowcat_clone.csv", index=False)
print("Saved submission_shadowcat_clone.csv")
